# Lab 05.1: Tensor Parallelism

Column-parallel and row-parallel linear layers, AllReduce communication modeling,
attention head distribution, and scaling efficiency across interconnects.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
from content.utils.benchmark import Timer
from content.utils.gpu_info import get_gpu_specs

## Column-Parallel Linear Layer

Split weight matrix along the column dimension — each GPU holds W[:, start:end] and computes a shard of the output.

In [ ]:
def column_parallel_linear(X, W, num_gpus):
    """Simulate column-parallel: split W along columns across GPUs."""
    assert W.shape[1] % num_gpus == 0
    shard_size = W.shape[1] // num_gpus
    outputs = []
    for i in range(num_gpus):
        W_shard = W[:, i * shard_size:(i + 1) * shard_size]
        outputs.append(X @ W_shard)
    return outputs  # Each GPU has partial output

# Demo: hidden_dim=1024, output_dim=4096, 4 GPUs
np.random.seed(42)
batch, seq, hidden, out_dim, n_gpus = 2, 128, 1024, 4096, 4
X = np.random.randn(batch * seq, hidden).astype(np.float32)
W = np.random.randn(hidden, out_dim).astype(np.float32)

shards = column_parallel_linear(X, W, n_gpus)
print(f'Input: {X.shape} | Weight: {W.shape}')
print(f'Each GPU output shard: {shards[0].shape}')
print(f'Concatenated = full output: {np.concatenate(shards, axis=1).shape}')

## Row-Parallel Linear Layer

Split weight matrix along the row dimension — each GPU holds W[start:end, :]. Input must be split correspondingly, then outputs are summed (AllReduce).

In [ ]:
def row_parallel_linear(X_shards, W, num_gpus):
    """Simulate row-parallel: split W along rows, each GPU has input shard."""
    assert W.shape[0] % num_gpus == 0
    shard_size = W.shape[0] // num_gpus
    partial_outputs = []
    for i in range(num_gpus):
        W_shard = W[i * shard_size:(i + 1) * shard_size, :]
        partial_outputs.append(X_shards[i] @ W_shard)
    # AllReduce: sum partial outputs
    return sum(partial_outputs)

# Use column-parallel output shards as input to row-parallel
W2 = np.random.randn(out_dim, hidden).astype(np.float32)
result = row_parallel_linear(shards, W2, n_gpus)
reference = (X @ W) @ W2
print(f'Row-parallel output: {result.shape}')
print(f'Max error vs reference: {np.max(np.abs(result - reference)):.6e}')

## AllReduce Communication Cost Model

Ring AllReduce transfers 2*(N-1)/N * message_size across N GPUs.

In [ ]:
def allreduce_time_us(msg_bytes, num_gpus, bw_gbps):
    """Estimate AllReduce latency (ring algorithm)."""
    bw_bytes_per_us = bw_gbps * 1e9 / 8 / 1e6  # GB/s -> bytes/us
    volume = 2 * (num_gpus - 1) / num_gpus * msg_bytes
    return volume / bw_bytes_per_us

# Compare interconnects
interconnects = {'NVLink (900 GB/s)': 900, 'PCIe 5.0 (64 GB/s)': 64, 'EFA (100 Gbps)': 12.5}
msg_sizes_mb = [1, 10, 50, 100, 500]

print(f'{"Size (MB)":<12}', end='')
for name in interconnects:
    print(f'{name:<22}', end='')
print()
print('-' * 78)
for mb in msg_sizes_mb:
    msg_bytes = mb * 1024 * 1024
    print(f'{mb:<12}', end='')
    for name, bw in interconnects.items():
        t = allreduce_time_us(msg_bytes, 8, bw * 8)  # bw in Gbps
        print(f'{t/1000:.2f} ms{"":<14}', end='')
    print()

## Attention Head Distribution

In tensor-parallel attention, heads are evenly split across GPUs. Each GPU computes its subset independently — no communication until the output projection.

In [ ]:
def distribute_attention_heads(num_heads, head_dim, num_gpus, batch_seq):
    """Show how attention heads map to GPUs."""
    assert num_heads % num_gpus == 0
    heads_per_gpu = num_heads // num_gpus
    kv_per_gpu_bytes = 2 * batch_seq * heads_per_gpu * head_dim * 2  # K+V, fp16
    print(f'Total heads: {num_heads} | Heads/GPU: {heads_per_gpu}')
    print(f'KV cache per GPU: {kv_per_gpu_bytes / 1024**2:.1f} MB (per layer, bs*seq={batch_seq})')
    for gpu in range(num_gpus):
        start = gpu * heads_per_gpu
        end = start + heads_per_gpu
        print(f'  GPU {gpu}: heads [{start}..{end-1}]')
    return heads_per_gpu

# Llama-70B: 64 heads, dim 128
distribute_attention_heads(num_heads=64, head_dim=128, num_gpus=8, batch_seq=4*2048)

## MLP Tensor Parallelism: Full Forward Pass

Standard transformer MLP: Column-parallel on first linear, Row-parallel on second. One AllReduce per MLP block.

In [ ]:
def tensor_parallel_mlp(X, W_gate, W_down, num_gpus):
    """Full TP MLP: column-parallel gate -> GeLU -> row-parallel down."""
    # Column-parallel first linear
    gate_shards = column_parallel_linear(X, W_gate, num_gpus)
    # Activation (independent per GPU)
    activated = [np.maximum(0, s) * (1 + np.tanh(np.sqrt(2/np.pi) * (s + 0.044715 * s**3)))/2
                 for s in gate_shards]  # Approximate GeLU
    # Row-parallel second linear (includes AllReduce sum)
    output = row_parallel_linear(activated, W_down, num_gpus)
    return output

W_gate = np.random.randn(hidden, out_dim).astype(np.float32) * 0.01
W_down = np.random.randn(out_dim, hidden).astype(np.float32) * 0.01
out = tensor_parallel_mlp(X[:16], W_gate, W_down, n_gpus)
print(f'MLP output shape: {out.shape} (1 AllReduce for {n_gpus} GPUs)')

## Scaling Efficiency Calculator

Compute efficiency = compute_time / (compute_time + comm_time) as we scale GPUs.

In [ ]:
def scaling_efficiency(model_params_B, seq_len, batch_size, num_gpus_list, interconnect_gbps):
    """Estimate TP scaling efficiency for different GPU counts."""
    # Approximate: 2 AllReduces per layer (MLP + Attention output)
    num_layers = int(model_params_B * 1e9 / (12 * 4096**2))  # rough estimate
    hidden = 4096 if model_params_B < 20 else 8192
    results = []
    for n in num_gpus_list:
        # Compute time scales ~linearly with 1/n
        flops = 2 * model_params_B * 1e9 * batch_size * seq_len
        gpu_tflops = 312e12  # A100 fp16
        compute_s = flops / (gpu_tflops * n) 
        # Comm: 2 allreduces/layer, msg = batch*seq*hidden*2 bytes
        msg_bytes = batch_size * seq_len * hidden * 2
        bw_bytes_s = interconnect_gbps * 1e9 / 8
        comm_per_layer = 2 * (2 * (n-1)/n * msg_bytes) / bw_bytes_s
        total_comm = comm_per_layer * num_layers
        eff = compute_s / (compute_s + total_comm)
        results.append((n, eff * 100, compute_s * 1000, total_comm * 1000))
    return results

print('=== 7B Model, batch=8, seq=2048 ===')
print(f'{"GPUs":<6}{"NVLink %":<12}{"PCIe %":<12}{"EFA %":<12}')
print('-' * 42)
for n in [1, 2, 4, 8]:
    nvl = scaling_efficiency(7, 2048, 8, [n], 900)[0]
    pci = scaling_efficiency(7, 2048, 8, [n], 64)[0]
    efa = scaling_efficiency(7, 2048, 8, [n], 100)[0]
    print(f'{n:<6}{nvl[1]:<12.1f}{pci[1]:<12.1f}{efa[1]:<12.1f}')

## Multi-GPU VRAM Calculator

Calculate per-GPU memory with tensor parallelism: model weights + KV cache + activations.

In [ ]:
def vram_per_gpu(params_B, num_layers, hidden, num_heads, head_dim,
                 num_gpus, batch, seq_len, dtype_bytes=2):
    """Calculate per-GPU VRAM breakdown with tensor parallelism."""
    # Weights: evenly split
    weight_bytes = params_B * 1e9 * dtype_bytes / num_gpus
    # KV cache: heads split across GPUs
    heads_per_gpu = num_heads // num_gpus
    kv_bytes = 2 * num_layers * batch * seq_len * heads_per_gpu * head_dim * dtype_bytes
    # Activations: batch * seq * hidden (not split for residual stream)
    act_bytes = batch * seq_len * hidden * dtype_bytes * 4  # ~4 activation tensors
    return {
        'weights_gb': weight_bytes / 1024**3,
        'kv_cache_gb': kv_bytes / 1024**3,
        'activations_gb': act_bytes / 1024**3,
        'total_gb': (weight_bytes + kv_bytes + act_bytes) / 1024**3
    }

models = [
    ('Llama-7B', 7, 32, 4096, 32, 128),
    ('Llama-70B', 70, 80, 8192, 64, 128),
    ('Llama-405B', 405, 126, 16384, 128, 128),
]

print(f'{"Model":<12}{"GPUs":<6}{"Weights":<10}{"KV Cache":<10}{"Acts":<8}{"Total":<8}{"Fits 80GB?"}')
print('-' * 64)
for name, params, layers, hidden, heads, hdim in models:
    for ng in [1, 2, 4, 8]:
        if heads % ng != 0:
            continue
        v = vram_per_gpu(params, layers, hidden, heads, hdim, ng, batch=4, seq_len=2048)
        fits = '✓' if v['total_gb'] < 80 else '✗'
        print(f'{name:<12}{ng:<6}{v["weights_gb"]:<10.1f}{v["kv_cache_gb"]:<10.1f}'
              f'{v["activations_gb"]:<8.1f}{v["total_gb"]:<8.1f}{fits}')

## Communication-Computation Overlap

Modern frameworks overlap AllReduce with computation. Measure the potential speedup.

In [ ]:
def overlap_analysis(model_params_B, num_gpus, bw_gbps, overlap_pct=0.7):
    """Show benefit of overlapping communication with compute."""
    res_no_overlap = scaling_efficiency(model_params_B, 2048, 8, [num_gpus], bw_gbps)[0]
    _, eff_no, compute_ms, comm_ms = res_no_overlap
    # With overlap: effective comm cost reduced
    effective_comm = comm_ms * (1 - overlap_pct)
    eff_overlap = compute_ms / (compute_ms + effective_comm) * 100
    return eff_no, eff_overlap, comm_ms, effective_comm

print('Communication-Computation Overlap Analysis (70B, 8 GPUs)')
print(f'{"Interconnect":<20}{"No Overlap":<14}{"70% Overlap":<14}{"Comm (ms)":<12}{"Eff Comm"}')
print('-' * 62)
for name, bw in [('NVLink', 900), ('PCIe 5.0', 64), ('EFA 100G', 100)]:
    no_ov, with_ov, comm, eff_comm = overlap_analysis(70, 8, bw)
    print(f'{name:<20}{no_ov:<14.1f}{with_ov:<14.1f}{comm:<12.2f}{eff_comm:.2f}')

## Summary

Key takeaways:
- **Column-parallel**: splits output dimension, no communication needed until row-parallel
- **Row-parallel**: splits input dimension, requires AllReduce to sum partial results
- **Attention TP**: heads distributed evenly, communication only at output projection
- **NVLink >> PCIe >> EFA** for TP efficiency — keep TP within a node
- **VRAM scales ~linearly** with 1/num_gpus for weights and KV cache
- **Overlap** can recover 50-70% of communication overhead in practice